# Step 1 · Data inspection

**Status:** Completed for available Swedish dev data; some organizer questions remain open.

Inspect the actual dataset, summarize imbalance and document preprocessing. No labels are used to repair spans. This notebook does not download or alter the raw data.

Start with a fresh runtime and execute cells in order. Read each section before running it. The original combined notebook and its outputs are retained as historical evidence.

## 1. Connect to the project
CPU is sufficient unless you explicitly enable new model inference. This uses the existing Drive project; it does not delete or recopy it.

In [ ]:
from pathlib import Path
import sys, os
try:
    from google.colab import drive
except ImportError:
    PROJECT = Path.cwd().resolve()
    if PROJECT.name == 'notebooks':
        PROJECT = PROJECT.parent
else:
    drive.mount('/content/drive')
    PROJECT = Path('/content/drive/MyDrive/semeval2027/semeval2027_colab')
assert (PROJECT/'src/semeval_task3/data.py').is_file(), f'Edit PROJECT to your project root: {PROJECT}'
assert (PROJECT/'notebooks/project_utils.py').is_file(), 'Use the organized project, including notebooks/project_utils.py.'
sys.path.insert(0, str(PROJECT/'src'))
sys.path.insert(0, str(PROJECT/'notebooks'))
os.environ['PYTHONPATH'] = str(PROJECT/'src')
from project_utils import (run_checked, new_run, save_json, read_scores, save_scores,
                           frozen, standardized_by_word, record_environment, save_evaluation, digest)
from semeval_task3.data import load_dataset
from semeval_task3.cv import Candidate, nested_word_cv
from semeval_task3.evaluation import score, word_bootstrap, read_submission
print('Project:', PROJECT)
print('Existing files stay in place. New outputs use unique directories under outputs/runs/.')

## 2. Inspect counts and schema

In [ ]:
from collections import Counter
import json
raw = load_dataset(PROJECT/'work/data/dev', with_gold=True, repair_spans=False)
dev = load_dataset(PROJECT/'work/data/dev', with_gold=True)
assert raw.keys == dev.keys and raw.gold == dev.gold
print('Languages:', sorted({u.language for u in dev.usages}))
print('Usages:', len(dev.usages), '| Words:', len({u.group for u in dev.usages}))
print('Positive:', sum(dev.gold.values()), '| Negative:', len(dev.gold)-sum(dev.gold.values()))
print('Years:', min(u.year for u in dev.usages), 'to', max(u.year for u in dev.usages))
print('Input example:', dev.usages[0])
print('Gold is separate:', dev.gold[dev.usages[0].key])

## 3. Counts and class imbalance by word and period

In [ ]:
summary = []
for group in sorted({u.group for u in dev.usages}):
    rows = [u for u in dev.usages if u.group == group]
    positive = sum(dev.gold[u.key] for u in rows)
    item = dict(language=group[0], word=group[1], usages=len(rows), positives=positive,
                positive_rate=positive/len(rows), periods=len({u.period_label for u in rows}))
    summary.append(item)
    print(f"{group[0]} {group[1]:12} n={len(rows):4} positive={positive:3} ({positive/len(rows):.1%})")
periods = []
for language, word, period in sorted({(u.language,u.word,u.period_label) for u in dev.usages}):
    rows = [u for u in dev.usages if (u.language,u.word,u.period_label)==(language,word,period)]
    periods.append(dict(language=language,word=word,period=period,n=len(rows),positives=sum(dev.gold[u.key] for u in rows)))
print('Word-period groups:', len(periods))

## 4. Audit the five repaired spans
 The loader retains original offsets in `original_span`. It repairs a whitespace-containing span only if there is exactly one case-insensitive headword-prefix token of the original span length. Ambiguous cases stay flagged. This heuristic is verified on the five current Swedish examples; it is not a general multilingual lemmatizer.

In [ ]:
repairs = []
for u in dev.usages:
    if u.original_span:
        a,b = u.original_span
        item = dict(language=u.language,word=u.word,sentence_id=u.sentence_id,
                    original=[a,b],repaired=[u.start,u.end],original_text=u.text[a:b],target=u.text[u.start:u.end])
        repairs.append(item)
        print(item)
print('Flagged before:', sum(bool(u.span_flags) for u in raw.usages))
print('Flagged after:', sum(bool(u.span_flags) for u in dev.usages))
RUN_DIR = new_run(PROJECT, 'step01_data')
save_json(RUN_DIR/'data_summary.json', dict(by_word=summary,by_word_period=periods,span_repairs=repairs))
record_environment(PROJECT,RUN_DIR/'manifest.json',purpose='Data inspection; raw files unchanged')
print('Saved:', RUN_DIR)

## Done when
Counts match the data card, original IDs/labels are unchanged, and all repairs are traceable. See `docs/DATA_AND_RULES.md` for unresolved questions. Next: Step 2.